In [1]:
from scipy import stats

# run model on 5 random seeds for reproducibility (seeds 7, 32, 33, 42, 63), obtain ROC-AUC values
horde_845 = [0.8738947345051566, 0.875890339231197, 0.8718010869424179, 0.881489918792639, 0.8767328248714876]
baseline = [0.8692354457812478, 0.869841302157107, 0.8639067314714455, 0.8699695098282144, 0.8650905149115553]

# Perform paired t-test
t_statistic, p_value = stats.ttest_rel(horde_845, baseline)

print(f"t-statistic: {t_statistic}, p-value: {p_value}")


t-statistic: 5.905475654950839, p-value: 0.004114894354808054


In [2]:
""" 
For each ablation of OR logits, load the ROC AUC values across 5 seeds and perform paired t-test against the baseline model (no OR logits).
Baseline directory: ../weighted_loss_HORDE_2/gs_lf_{n_or}_OR_logits_layernorm_roc_auc_score
for n_or in [0, 5, 10, 20, 50, 100, 400, 845]

In each, there is a .txt file named {seed}_eval.txt for each seed with the following format:
Best val roc_auc_score: 0.871048254670169
Val roc_auc_score: 0.871048254670169
Test roc_auc_score: 0.8722927786008038
Val prc_auc_score: 0.2929044165327258
Test prc_auc_score: 0.2878370815337783

We need to load the Test roc_auc_score values for each seed to compute the t-test, and report the p-value for each ablation, and the mean and std of the Test roc_auc_score values for each ablation.
"""

import os
import numpy as np
from scipy import stats

# Define parameters
n_or_values = [0, 5, 10, 20, 50, 100, 400, 845]
seeds = [7, 17, 24, 32, 42, 63]
base_dir = "../weighted_loss_HORDE_2"

# Function to load Test roc_auc_score from eval file
def load_test_roc_auc(filepath):
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('Test roc_auc_score:'):
                return float(line.split(':')[1].strip())
    return None

# Collect data for each ablation
results = {}

for n_or in n_or_values:
    dir_path = f"{base_dir}/gs_lf_{n_or}_OR_logits_layernorm_roc_auc_score"
    scores = []
    
    for seed in seeds:
        eval_file = f"{dir_path}/{seed}_eval.txt"
        if os.path.exists(eval_file):
            score = load_test_roc_auc(eval_file)
            if score is not None:
                scores.append(score)
    
    if scores:
        results[n_or] = scores

# Perform statistical tests and report results (uncorrected)
# Use baseline from cell 0 (seeds 7, 32, 33, 42, 63)
print("Ablation Study Results (uncorrected paired t-tests)")
print("=" * 50)

for n_or in n_or_values:
    if n_or in results:
        scores = results[n_or]
        mean_score = np.mean(scores)
        std_score = np.std(scores, ddof=1)
        
        print(f"\nn_or = {n_or}:")
        print(f"  Mean Test ROC-AUC: {mean_score:.6f}")
        print(f"  Std Test ROC-AUC: {std_score:.6f}")
        print(f"  Scores: {scores}")
        
        if n_or != 0 and baseline is not None:
            if len(scores) == len(baseline):
                t_statistic, p_value = stats.ttest_rel(scores, baseline)
                print(f"  t-statistic vs baseline: {t_statistic:.6f}")
                print(f"  p-value vs baseline: {p_value:.6f}")
            else:
                print(f"  Warning: Cannot perform t-test - different number of scores")
        elif n_or == 0:
            print(f"  (Baseline model)")

Ablation Study Results (uncorrected paired t-tests)

n_or = 0:
  Mean Test ROC-AUC: 0.870461
  Std Test ROC-AUC: 0.004754
  Scores: [0.8692354457812478, 0.878170201753475, 0.869841302157107, 0.8699695098282144, 0.8650905149115553]
  (Baseline model)

n_or = 5:
  Mean Test ROC-AUC: 0.872883
  Std Test ROC-AUC: 0.003899
  Scores: [0.8751584910044962, 0.8694710620633621, 0.867886814537443, 0.875624726372334, 0.8762763335160499]
  t-statistic vs baseline: 2.838864
  p-value vs baseline: 0.046922

n_or = 10:
  Mean Test ROC-AUC: 0.871854
  Std Test ROC-AUC: 0.002130
  Scores: [0.869440384350885, 0.870950807638334, 0.8751428377261953, 0.8712763262747809, 0.872457647867637]
  t-statistic vs baseline: 1.964204
  p-value vs baseline: 0.120970

n_or = 20:
  Mean Test ROC-AUC: 0.873323
  Std Test ROC-AUC: 0.003034
  Scores: [0.8760440215399395, 0.8706662371665449, 0.8708343007381354, 0.8771063426011758, 0.8719653248625249]
  t-statistic vs baseline: 4.670132
  p-value vs baseline: 0.009517

n_or 

In [3]:
"""
Apply Benjamini-Hochberg FDR correction for multiple comparisons across the 7 ablation 
t-tests (each ablation vs baseline). This addresses Reviewer #1's concern about whether 
p-values in Table S1 were corrected for multiple comparisons.
"""
from statsmodels.stats.multitest import multipletests

# Collect raw p-values for all non-baseline ablations (using cell 0 baseline)
raw_pvalues = []
ablation_keys = []

for n_or in n_or_values:
    if n_or == 0 or n_or not in results:
        continue
    scores = results[n_or]
    if len(scores) == len(baseline):
        t_stat, p_val = stats.ttest_rel(scores, baseline)
        raw_pvalues.append(p_val)
        ablation_keys.append(n_or)

# Apply Benjamini-Hochberg FDR correction
rejected, pvals_corrected, _, _ = multipletests(raw_pvalues, alpha=0.05, method='fdr_bh')

# Report results
print("Ablation Study Results (with Benjamini-Hochberg FDR correction)")
print("=" * 70)
print(f"Number of comparisons: {len(raw_pvalues)}")
print(f"\n{'# ORs':<10} {'t-stat':<12} {'Raw p':<12} {'BH q-value':<12} {'Sig (FDR<0.05)'}")
print("-" * 60)

for idx, n_or in enumerate(ablation_keys):
    scores = results[n_or]
    t_stat, _ = stats.ttest_rel(scores, baseline)
    sig = "*" if rejected[idx] else ""
    print(f"{n_or:<10} {t_stat:<12.4f} {raw_pvalues[idx]:<12.6f} {pvals_corrected[idx]:<12.6f} {rejected[idx]}{sig}")

Ablation Study Results (with Benjamini-Hochberg FDR correction)
Number of comparisons: 7

# ORs      t-stat       Raw p        BH q-value   Sig (FDR<0.05)
------------------------------------------------------------
5          2.8389       0.046922     0.065690     False
10         1.9642       0.120970     0.141132     False
20         4.6701       0.009517     0.026556     True*
50         3.4832       0.025279     0.044238     True*
100        0.6348       0.560028     0.560028     False
400        5.0516       0.007223     0.026556     True*
845        4.4346       0.011381     0.026556     True*


In [4]:
"""
Jonckheere-Terpstra trend test: tests the ordered alternative hypothesis that 
percept prediction performance monotonically increases with the number of OR 
activations. This is a single test (no multiple comparison issue) that directly 
addresses the reviewer's concern about whether the scaling trend is robust.

H0: Performance is the same across all #OR levels
H1: Performance increases with #OR (ordered alternative)
"""

def jonckheere_terpstra_test(groups, n_permutations=10000):
    """
    Jonckheere-Terpstra test for ordered alternatives.
    Groups should be ordered from expected lowest to expected highest.
    Test statistic J = sum of Mann-Whitney U statistics for all i < j pairs.
    P-value computed via permutation.
    """
    k = len(groups)
    
    # Compute observed J statistic
    def compute_J(groups):
        J = 0
        for i in range(k):
            for j in range(i + 1, k):
                for xi in groups[i]:
                    for xj in groups[j]:
                        if xj > xi:
                            J += 1
                        elif xj == xi:
                            J += 0.5
        return J
    
    J_obs = compute_J(groups)
    
    # Permutation test for p-value
    all_values = np.concatenate(groups)
    group_sizes = [len(g) for g in groups]
    
    count_ge = 0
    np.random.seed(42)
    for _ in range(n_permutations):
        perm = np.random.permutation(all_values)
        perm_groups = []
        idx = 0
        for size in group_sizes:
            perm_groups.append(perm[idx:idx + size])
            idx += size
        J_perm = compute_J(perm_groups)
        if J_perm >= J_obs:
            count_ge += 1
    
    p_value = count_ge / n_permutations
    return J_obs, p_value

# Ordered groups: baseline (0 ORs) through 845 ORs
# Use the same scores loaded in cell 1
ordered_n_ors = [0, 5, 10, 20, 50, 100, 400, 845]
ordered_groups = []
for n_or in ordered_n_ors:
    if n_or == 0:
        ordered_groups.append(np.array(baseline))
    elif n_or in results:
        ordered_groups.append(np.array(results[n_or]))

print("Jonckheere-Terpstra Test for Monotonic Trend")
print("=" * 50)
print(f"Ordered groups: {ordered_n_ors}")
print(f"Group sizes: {[len(g) for g in ordered_groups]}")
print(f"Group means: {[f'{np.mean(g):.4f}' for g in ordered_groups]}")

J_obs, p_value = jonckheere_terpstra_test(ordered_groups, n_permutations=10000)
print(f"\nJ statistic: {J_obs}")
print(f"P-value (one-sided, 10000 permutations): {p_value:.4f}")
print(f"Significant (p < 0.05): {p_value < 0.05}")

Jonckheere-Terpstra Test for Monotonic Trend
Ordered groups: [0, 5, 10, 20, 50, 100, 400, 845]
Group sizes: [5, 5, 5, 5, 5, 5, 5, 5]
Group means: ['0.8676', '0.8729', '0.8719', '0.8733', '0.8727', '0.8692', '0.8724', '0.8762']

J statistic: 449
P-value (one-sided, 10000 permutations): 0.0103
Significant (p < 0.05): True
